# 第 3 章｜Tool Calling

依序執行每一格；可修改標示的參數後重跑。

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from course_utils import *
print("教材根目錄：", ROOT)
config = require_cillm_config()
print("執行模式：CILLM API（必要）")
print("GPT-OSS 模型：", config["model"])

> **執行模式 Hint**
>
> - 本教材每章都必須設定 `CILLM_API_KEY` 與 `CILLM_BASE_URL`，並呼叫 `openai/gpt-oss-120b`。
> - 缺少設定時會立即停止，不會以 mock 回覆取代真實模型。
> - 圖片解析另使用 `google/gemma-4-31b-it`，但沿用相同 CILLM API key。

> **Tool Hint**
>
> - 數學題可改座位數或載客率，預期選擇 `math_tool`。
> - 圖片路徑可換成延誤看板、行李標籤或機坪照片，預期選擇 `image_tool`。
> - Excel 問題可改成平均值、最大延誤或依部門分組，預期顯示不同 pandas 程式碼與結果。

## Tool Registry

In [ ]:
for name, meta in TOOL_REGISTRY.items(): print(f"- {name}: {meta['description']}")
MAX_TOKENS = 800  # 調低可觀察回答或程式碼可能不完整

## 數學 Tool：模型選擇 Tool 後執行真實運算。

In [ ]:
USER_REQUEST = "一架飛機有 312 個座位，載客率是 87%，大約有多少位旅客"
MAX_TOKENS = 800  # 調低可觀察回答或程式碼可能不完整
router_note = ask_gpt_oss("可用工具有 math_tool、image_tool、excel_python_tool。座位數乘載客率的問題需要哪個工具？只回答工具名稱與理由。")
print("GPT-OSS Tool 路由：", router_note)

## 數學 Tool：模型選擇 Tool 後執行真實運算。

In [ ]:
USER_REQUEST = "一架飛機有 312 個座位，載客率是 87%，大約有多少位旅客"
result = math_tool(312, "*", 0.87)
answer = f"約有 {round(result)} 位旅客。"
print_execution_trace(question=USER_REQUEST, tool="✓ math_tool", tool_result=result, answer=answer)

## 圖片 Tool：依問題呼叫 Gemma。

In [ ]:
USER_REQUEST = "這張登機證是哪一個航班與登機門？"
image_result = analyze_image(ROOT / "data/images/mock_boarding_pass.png", USER_REQUEST)
show(image_result)
print_execution_trace(question=USER_REQUEST, tool="✓ image_tool", tool_result=image_result["content"], answer=image_result["content"])

## Excel Python Tool

完整顯示問題、模型產生的程式碼、執行結果。此限制僅供教學，不是 production sandbox。

In [ ]:
USER_REQUEST = "找出延誤超過 120 分鐘的航班"
# GPT-OSS 可產生此段；目前保留可重現的程式，方便先觀察執行與安全限制。
GENERATED_CODE = """df = pd.read_excel(excel_path)\nresult = df.loc[df['delay_minutes'] > 120, ['flight','delay_minutes']].to_dict('records')"""
print("使用者問題：", USER_REQUEST)
print("GPT-OSS 產生的 Python 程式碼：\n", GENERATED_CODE)
tool_result = safe_excel_python(ROOT / "data/excel/flight_delays.xlsx", GENERATED_CODE)
print("實際執行結果："); show(tool_result)
answer = f"共有 {len(tool_result)} 個航班延誤超過 120 分鐘。"
print("最後回答：", answer)
print_execution_trace(question=USER_REQUEST, tool="✓ excel_python_tool", tool_result=tool_result, answer=answer)

### 小練習

將門檻改成 90，或暫時從 `TOOL_REGISTRY` 移除一個 Tool。